<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-08/AI_Modul_8.3_Praktikum_Evaluasi_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 8.3: Praktikum Evaluasi Model
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Menghitung matriks metrik evaluasi regresi menyeluruh (RMSE, MAE, MAPE, $R^2$) pada data uji independen.
2. Mengonstruksi visualisasi diagnostik tingkat lanjut: *Actual vs Predicted Plot*, *Residual vs Fitted Plot*, dan *Normal Q-Q Plot*.
3. Menguji asumsi statistik residual (uji normalitas Shapiro-Wilk dan uji homoskedastisitas).
4. Melakukan uji stres ketahanan model (*Robustness Stress Testing*) terhadap gangguan derau sensor lapangan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Konfigurasi seed komputasi
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Perangkat Komputasi Aktif: {device}")


## 1. Penyiapan Dataset Terpadu & Pelatihan Model Baseline
Kita menyiapkan dataset spektral dan sensor IoT kelapa sawit (600 sampel, 124 fitur) serta melatih arsitektur JST 3-lapisan [124, 64, 32, 1] hingga konvergen.


In [ ]:
n_samples = 600
n_spectral = 120
afdeling_ids = np.repeat(np.arange(1, 11), n_samples // 10)

# Fitur Spektral & Sensor
wavelengths = np.linspace(400, 1000, n_spectral)
X_spectral = np.zeros((n_samples, n_spectral))
for i in range(n_samples):
    base_sig = 0.15 + 0.35 * np.exp(-((wavelengths - 550)**2) / (2 * 45**2)) +                       0.50 * (1.0 / (1.0 + np.exp(-(wavelengths - 700) / 20)))
    noise = np.random.normal(0, 0.02, n_spectral)
    X_spectral[i] = base_sig * np.random.uniform(0.85, 1.15) + noise

pH = np.random.normal(5.2, 0.4, (n_samples, 1))
nitrogen = np.random.normal(120.0, 25.0, (n_samples, 1))
moisture = np.random.uniform(30.0, 70.0, (n_samples, 1))
rainfall = np.random.uniform(100.0, 350.0, (n_samples, 1))
X_sensor = np.hstack([pH, nitrogen, moisture, rainfall])
X_raw = np.hstack([X_spectral, X_sensor])

y_raw = 19.5 + 4.5 * X_spectral[:, 65] - 0.015 * np.abs(pH[:, 0] - 5.5)**2 +         0.012 * (nitrogen[:, 0] / 10.0) + np.random.normal(0, 0.35, n_samples)
y_raw = y_raw.reshape(-1, 1)

# Partisi Spasial
train_mask = np.isin(afdeling_ids, [1, 2, 3, 4, 5, 6])
val_mask   = np.isin(afdeling_ids, [7, 8])
test_mask  = np.isin(afdeling_ids, [9, 10])

# Standarisasi
mu_tr = np.mean(X_raw[train_mask], axis=0, keepdims=True)
std_tr = np.std(X_raw[train_mask], axis=0, keepdims=True) + 1e-8

X_train_norm = (X_raw[train_mask] - mu_tr) / std_tr
X_val_norm   = (X_raw[val_mask] - mu_tr) / std_tr
X_test_norm  = (X_raw[test_mask] - mu_tr) / std_tr

y_train = y_raw[train_mask]
y_val   = y_raw[val_mask]
y_test  = y_raw[test_mask]

class AgroDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(AgroDataset(X_train_norm, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(AgroDataset(X_test_norm, y_test), batch_size=32, shuffle=False)

# Arsitektur JST
class OilPalmYieldMLP(nn.Module):
    def __init__(self, in_features=124, hidden1=64, hidden2=32, out_features=1):
        super(OilPalmYieldMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(p=0.15),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(p=0.10),
            nn.Linear(hidden2, out_features)
        )
    def forward(self, x):
        return self.net(x)

model = OilPalmYieldMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.MSELoss()

# Pelatihan Ringkas 35 Epoch
model.train()
for epoch in range(35):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

print("Model berhasil dilatih dan siap diaudit!")


## 2. Kalkulasi Matriks Metrik Evaluasi Komprehensif
Kita menghitung metrik kuantitatif pada himpunan data uji independen (Afdeling 9 dan 10):
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Mean Absolute Percentage Error (MAPE)
- Koefisien Determinasi ($R^2$)
- Persentase Sampel Memenuhi Toleransi Industri ($\pm 0.8\%$ CPO)


In [ ]:
model.eval()
y_true_list = []
y_pred_list = []

with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        preds = model(bx)
        y_true_list.extend(by.numpy().flatten())
        y_pred_list.extend(preds.cpu().numpy().flatten())

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)
residuals = y_true - y_pred

# Kalkulasi Metrik
rmse = np.sqrt(np.mean(residuals**2))
mae = np.mean(np.abs(residuals))
mape = np.mean(np.abs(residuals / y_true)) * 100.0

ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2 = 1.0 - (ss_res / ss_tot)

tolerance = 0.8
in_tolerance_pct = np.mean(np.abs(residuals) <= tolerance) * 100.0

print("=================================================================")
print("             AUDIT METRIK EVALUASI MODEL DATA UJI                ")
print("=================================================================")
print(f"Root Mean Squared Error (RMSE)   : {rmse:.4f} % Rendemen")
print(f"Mean Absolute Error (MAE)        : {mae:.4f} % Rendemen")
print(f"Mean Absolute Percentage Error   : {mape:.2f} %")
print(f"Koefisien Determinasi (R2)       : {r2:.4f}")
print(f"Kepatuhan Toleransi Industri (±0.8%): {in_tolerance_pct:.1f} % Sampel")
print("=================================================================")


## 3. Visualisasi Panel Diagnostik Regresi dan Residual
Kita menyusun visualisasi 3-panel:
1. **Actual vs Predicted:** Memeriksa keselarasan terhadap garis ideal 1:1.
2. **Residual vs Predicted:** Menguji asumsi homoskedastisitas (apakah varians galat konstan).
3. **Q-Q Plot Residual:** Menguji apakah galat berdistribusi normal Gaussian.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Actual vs Predicted
min_val, max_val = np.min(y_true) - 0.5, np.max(y_true) + 0.5
axes[0].scatter(y_true, y_pred, color='dodgerblue', alpha=0.8, edgecolors='k', s=55, label='Sampel Uji (N=120)')
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal 1:1')
axes[0].fill_between([min_val, max_val], [min_val - tolerance, max_val - tolerance], 
                     [min_val + tolerance, max_val + tolerance], color='gray', alpha=0.15, label=f'Toleransi ±{tolerance}%')
axes[0].set_title('A. Plot Aktual vs Prediksi Rendemen CPO', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Rendemen Aktual (%)')
axes[0].set_ylabel('Rendemen Prediksi (%)')
axes[0].set_xlim(min_val, max_val)
axes[0].set_ylim(min_val, max_val)
axes[0].legend(loc='upper left')

# Panel 2: Residual vs Predicted
axes[1].scatter(y_pred, residuals, color='goldenrod', alpha=0.8, edgecolors='k', s=55)
axes[1].axhline(0, color='red', linestyle='--', lw=1.8)
axes[1].axhline(2 * np.std(residuals), color='blue', linestyle=':', label='+2σ Batas')
axes[1].axhline(-2 * np.std(residuals), color='blue', linestyle=':', label='-2σ Batas')
axes[1].set_title('B. Plot Residual vs Nilai Prediksi (Homoskedastisitas)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Rendemen Prediksi (%)')
axes[1].set_ylabel('Galat Residual (e = y - y_pred)')
axes[1].legend(loc='lower left')

# Panel 3: Normal Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('C. Normal Q-Q Plot Galat Residual', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Kuantil Teoretis Gaussian')
axes[2].set_ylabel('Kuantil Sampel Residual')

plt.tight_layout()
plt.show()


## 4. Uji Hipotesis Statistik Residual
Kita melakukan verifikasi statistik formal:
- **Uji Normalitas Shapiro-Wilk:** Menguji $H_0$: Galat berdistribusi normal.
- **Korelasi Rank Spearman:** Menguji ketergantungan absolut residual terhadap nilai prediksi (Heteroskedastisitas).


In [ ]:
shapiro_stat, shapiro_p = stats.shapiro(residuals)
spearman_corr, spearman_p = stats.spearmanr(np.abs(residuals), y_pred)

print("=================================================================")
print("             UJI ASUMSI STATISTIK RESIDUAL MODEL                 ")
print("=================================================================")
print(f"Uji Normalitas Shapiro-Wilk : Statistik = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("-> KESIMPULAN: Residual berdistribusi NORMAL (Asumsi Terpenuhi).")
else:
    print("-> KESIMPULAN: Terdapat deviasi minor dari distribusi normal murni.")

print("")
print(f"Uji Homoskedastisitas Spearman: Korelasi = {spearman_corr:.4f}, p-value = {spearman_p:.4f}")
if spearman_p > 0.05:
    print("-> KESIMPULAN: Residual bersifat HOMOSKEDASTIK (Varians Galat Konstan).")
else:
    print("-> KESIMPULAN: Terdeteksi pola varians non-konstan.")
print("=================================================================")


## 5. Uji Stres Ketahanan Model (*Robustness Stress Testing*)
Kita menguji degradasi performa model ketika fitur spektral lapangan diinjeksi derau Gaussian dengan magnitudo $\sigma \in [0.02, 0.05, 0.10, 0.20]$.


In [ ]:
noise_levels = [0.00, 0.02, 0.05, 0.10, 0.20]
stress_rmse = []

model.eval()
with torch.no_grad():
    for sigma in noise_levels:
        cur_preds = []
        for bx, by in test_loader:
            if sigma > 0.0:
                noise = torch.randn_like(bx) * sigma
                bx_eval = (bx + noise).to(device)
            else:
                bx_eval = bx.to(device)
            p = model(bx_eval)
            cur_preds.extend(p.cpu().numpy().flatten())
            
        cur_res = y_true - np.array(cur_preds)
        cur_rmse = np.sqrt(np.mean(cur_res**2))
        stress_rmse.append(cur_rmse)

# Visualisasi Degradasi Kinerja
plt.figure(figsize=(8, 4))
plt.plot(noise_levels, stress_rmse, 'ro-', lw=2, markersize=6)
plt.axhline(rmse, color='green', linestyle='--', label=f'Baseline Bersih (RMSE = {rmse:.3f})')
plt.title('Kurva Degradasi Ketahanan Model terhadap Injeksi Derau Sensor', fontsize=11, fontweight='bold')
plt.xlabel('Tingkat Deviasi Derau (Sigma)')
plt.ylabel('Root Mean Squared Error (RMSE)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Degradasi RMSE pada Derau Ekstrem (Sigma=0.20): +{stress_rmse[-1] - rmse:.4f} % Rendemen")
print("[OK] Sesi Praktikum Evaluasi & Diagnostik Kinerja Selesai!")
